# BRIO with 32 candidates: candidate generation, preprocessing, training with several seeds

Steps:
1. Check the existing 16-candidate files for gibberish candidates.
2. Small test to choose the diversity penalty for 32 candidates.
3. Generate 32 candidates for test, val and train (resumable, saved on Drive).
4. Preprocess into the BRIO dataset format and save it to Drive as a zip.
5. Train with `--max_num 32`, one run per seed with `--seed`.

During generation, an article with more than 1 candidate at ROUGE = 0 (gibberish) is generated again with a lower diversity penalty. Every such article is written to `<split>.out.regen_log.jsonl` next to the candidate file.

In [ ]:
# Install condacolab. The runtime restarts after this cell, that is normal.
!pip install -q condacolab
import condacolab
condacolab.install()

### Env

In [ ]:
# Clone the repo and mount Google Drive.
!git clone https://github.com/alvrian/BRIO.git

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

%cd BRIO

In [ ]:
# Same env setup as BRIO_1_1.ipynb, then the fine-tuned IndoBART and the Liputan6 source/target files.
! conda create --name env --file spec-file.txt
! conda run --no-capture-output -n env pip install -r requirements.txt
! conda run -n env python -c "import nltk;nltk.download('punkt')"
! git clone https://github.com/neulab/compare-mt.git
! cd compare-mt && conda run --no-capture-output -n env pip install -r requirements.txt
! cd compare-mt && conda run -n env python setup.py install
print("done setting up env")
! cp /content/drive/MyDrive/BRIO/indobart-liputan6-finetuned.zip .
! unzip -q indobart-liputan6-finetuned.zip
! rm indobart-liputan6-finetuned.zip
print("done copying indobart finetuned, start copying dataset...")
! cp /content/drive/MyDrive/BRIO/Dataset/liputan6_converted.zip .
! unzip -q liputan6_converted.zip
! rm liputan6_converted.zip
! cp /content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.source ./liputan6_converted/canonical
! cp /content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.target ./liputan6_converted/canonical
import os
os.makedirs('./cache', exist_ok=True)
os.makedirs('./result', exist_ok=True)
print("all done")

### Settings

In [ ]:
import os

NUM_CANDS = 32            # candidates per article
GEN_BATCH_SIZE = 8        # articles per generation batch; lower it if generation runs out of GPU memory
DIVERSITY_PENALTY = 0.5   # EDIT after the small test in step 2

# Source/target files (local, unzipped above) and the fine-tuned IndoBART that generates the candidates.
DATA_DIR = "./liputan6_converted/canonical"
MODEL_DIR = "./indobart-liputan6-finetuned"

# Drive folder for the new 32-candidate files, and the zip of the preprocessed dataset.
CAND_DIR = "/content/drive/MyDrive/BRIO/liputan6_candidates_32/canonical"
DATASET_ZIP = "/content/drive/MyDrive/BRIO/Dataset/liputan6_32.zip"

# Existing 16-candidate files, only used by the check in step 1.
OLD_CAND_DIR = "/content/drive/MyDrive/BRIO/liputan6_candidates/canonical"

# Checkpoint folder per seed: CKPT_BASE + seed.
CKPT_BASE = "/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-1_32_seed"

for split in ["train", "val", "test"]:
    for ext in ["source", "target"]:
        p = os.path.join(DATA_DIR, split + "." + ext)
        print("found  " if os.path.exists(p) else "MISSING", p)

### 1. Check the existing 16-candidate files

This shows how many candidates in the files you already trained on have ROUGE = 0, and at which candidate positions.

In [ ]:
# Report for each existing 16-candidate file on Drive (skipped if the file is not there).
for split in ["test", "val", "train"]:
    out_file = os.path.join(OLD_CAND_DIR, split + ".out")
    tgt_file = os.path.join(DATA_DIR, split + ".target")
    if os.path.exists(out_file):
        print("=====", split)
        !conda run --no-capture-output -n env python check_candidates.py --out_file "{out_file}" --tgt_file "{tgt_file}" --num_cands 16 --show 1
    else:
        print("not found:", out_file)

### 2. Small test: choose the diversity penalty

With 32 beam groups the diversity penalty adds up over more groups than with 16, so the later candidates may turn into gibberish at 0.5. This cell generates 32 candidates for the first 48 validation articles with three penalty values and reports each one.

How to read the report:
- "Candidates with ROUGE = 0" should be close to 0%. If it is high, the penalty is too strong.
- "Distinct candidates per article" should stay close to 32. If it drops a lot, the penalty is too weak and the candidates are duplicates.

Pick the largest penalty with almost no ROUGE = 0 candidates and set `DIVERSITY_PENALTY` in the Settings cell.

In [ ]:
# Generate 32 candidates for 48 validation articles with three penalties, without regeneration, then report.
for penalty in [0.5, 0.2, 0.1]:
    out_file = "./penalty_test/p%s/val.out" % penalty
    print("===== diversity penalty", penalty)
    !conda run --no-capture-output -n env python gen_candidate.py --dataset liputan6 --src_dir "{DATA_DIR}/val.source" --tgt_dir "{out_file}" --model_dir "{MODEL_DIR}" --num_cands {NUM_CANDS} --batch_size {GEN_BATCH_SIZE} --diversity_penalty {penalty} --num_samples 48 --no_rouge_check
    !conda run --no-capture-output -n env python check_candidates.py --out_file "{out_file}" --tgt_file "{DATA_DIR}/val.target" --num_cands {NUM_CANDS} --show 1

### 3. Generate 32 candidates

The output is written to `CAND_DIR` on Drive. If Colab disconnects, run the same cell again: it continues from the last article saved on Drive. The train split is the long one (193,883 articles).

In [ ]:
# Helper: generate the candidates of one split, with the ROUGE = 0 check and regeneration switched on.
def generate_split(split, sync_every):
    !conda run --no-capture-output -n env python gen_candidate.py --dataset liputan6 --src_dir "{DATA_DIR}/{split}.source" --tgt_dir "{CAND_DIR}/{split}.out" --model_dir "{MODEL_DIR}" --num_cands {NUM_CANDS} --batch_size {GEN_BATCH_SIZE} --diversity_penalty {DIVERSITY_PENALTY} --sync_every {sync_every}

print("NUM_CANDS =", NUM_CANDS, "| DIVERSITY_PENALTY =", DIVERSITY_PENALTY)

In [ ]:
# Test split (10,972 articles).
generate_split("test", 100)

In [ ]:
# Validation split (10,972 articles).
generate_split("val", 100)

In [ ]:
# Train split (193,883 articles). The file is copied to Drive every 500 batches because it becomes large.
generate_split("train", 500)

In [ ]:
# Report for the new 32-candidate files: ROUGE = 0 candidates per position and distinct candidates.
for split in ["test", "val", "train"]:
    print("=====", split)
    !conda run --no-capture-output -n env python check_candidates.py --out_file "{CAND_DIR}/{split}.out" --tgt_file "{DATA_DIR}/{split}.target" --num_cands {NUM_CANDS} --show 1

In [ ]:
# Articles that were regenerated, and how many of them are still above the limit after all retries.
import pandas as pd
for split in ["test", "val", "train"]:
    log_file = os.path.join(CAND_DIR, split + ".out.regen_log.jsonl")
    if os.path.exists(log_file):
        log = pd.read_json(log_file, lines=True)
        print(split, "| regenerated:", len(log), "| not fixed:", int((~log["fixed"]).sum()))
    else:
        print(split, "| no article needed regeneration")

### 4. Preprocess into the BRIO format

`preprocess.py` reads `.source`, `.target`, `.out` and their `.tokenized` copies from one folder, scores every candidate with ROUGE and writes one json file per article.

In [ ]:
# Put the candidate files next to the source/target files and make the .tokenized copies
# (no Stanford tokenizer for Indonesian, same as before).
import shutil
for split in ["train", "val", "test"]:
    shutil.copyfile(os.path.join(CAND_DIR, split + ".out"), os.path.join(DATA_DIR, split + ".out"))
    for ext in ["source", "target", "out"]:
        shutil.copyfile(os.path.join(DATA_DIR, split + "." + ext), os.path.join(DATA_DIR, split + "." + ext + ".tokenized"))
print("files ready")

In [ ]:
# Build ./liputan6/canonical/{train,val,test} with 32 candidates per article.
for split in ["test", "val", "train"]:
    !conda run --no-capture-output -n env python preprocess.py --src_dir "{DATA_DIR}" --tgt_dir ./liputan6/canonical --split {split} --cand_num {NUM_CANDS} --dataset liputan6 -l

# test.source / test.target are needed by main.py -e
! cp {DATA_DIR}/test.source ./liputan6/canonical/
! cp {DATA_DIR}/test.target ./liputan6/canonical/
! cp {DATA_DIR}/test.target ./liputan6/

In [ ]:
# Save the preprocessed dataset to Drive as one zip, so later sessions can skip steps 2 to 4.
! cd liputan6 && zip -q -r ../liputan6_32.zip .
! cp liputan6_32.zip "{DATASET_ZIP}"
print("saved to", DATASET_ZIP)

### 5. Training with 32 candidates, one run per seed

In a new session, run the env cell, the Settings cell and the restore cell below, then the training cell of the seed you want.

`--seed` and `--max_num` now override `config.py`. Each seed writes to its own checkpoint folder on Drive.

In [ ]:
# New session only: restore the preprocessed 32-candidate dataset from Drive into ./liputan6
import zipfile
if not os.path.exists("./liputan6/canonical/train"):
    ! cp "{DATASET_ZIP}" .
    with zipfile.ZipFile("./liputan6_32.zip", "r") as zip_ref:
        zip_ref.extractall("./liputan6")
print("train files:", len(os.listdir("./liputan6/canonical/train")))

In [ ]:
# Seed 1 (the default seed in config.py)
SEED = 970903
!conda run --no-capture-output -n env python main.py \
    --cuda --gpuid 0 --config liputan6 -l \
    --seed {SEED} --max_num {NUM_CANDS} \
    --checkpoint_save_dir {CKPT_BASE}{SEED}

In [ ]:
# Seed 2
SEED = 70905
!conda run --no-capture-output -n env python main.py \
    --cuda --gpuid 0 --config liputan6 -l \
    --seed {SEED} --max_num {NUM_CANDS} \
    --checkpoint_save_dir {CKPT_BASE}{SEED}

In [ ]:
# Seed 3
SEED = 81215
!conda run --no-capture-output -n env python main.py \
    --cuda --gpuid 0 --config liputan6 -l \
    --seed {SEED} --max_num {NUM_CANDS} \
    --checkpoint_save_dir {CKPT_BASE}{SEED}

### Notes

- **Out of GPU memory during training:** 32 candidates use about twice the memory of 16. Add `--batch_size 1 --accumulate_step 16` to the training command. This keeps the same number of articles per optimizer step as the default (batch size 2, accumulate 8).
- **Evaluation with `main.py -e`:** add `--max_num 32` so the reranking evaluation uses all 32 candidates.
- **ROUGE, BERTScore and the Gemini judge:** add the new `model_generation.bin` paths to `BRIO_CKPTS` in `BRIO_bertscore.ipynb` / `BRIO_llm_judge.ipynb`.